# 02C 数据清洗与特征构造

> 🟢 **Level A · 必须掌握** | 完成标准：`raw table → audit → clean → construct features → select columns → ML-ready table`。

本节开始使用公开 COFSpace 的真实 CoRE-COF CO₂ 1 bar 数据。

In [ ]:
import pandas as pd
url='https://raw.githubusercontent.com/gokhanonderaksu/COFSpace/main/OnlyCoRECOF%20-%20Feature%20Sets/CoRECOF%20-%20CO2%20-%201%20BAR.csv'
df=pd.read_csv(url)
display(df.head())
display(pd.DataFrame({'dtype':df.dtypes.astype(str),'missing':df.isna().sum(),'missing_%':(100*df.isna().mean()).round(2),'unique':df.nunique()}))
print('duplicates =',df.duplicated().sum())


## Feature / target / provenance
`CO2-1 bar (mol/kg)` 是 target。PLD、LCD、surface area、porosity 和元素比例是候选 features；ID、DOI、filename 应保留用于追踪，但通常不是模型输入。

In [ ]:
target='CO2-1 bar (mol/kg)'
features=['PLD (Å)','LCD (Å)','Sacc (m2/g-1)','Porosity','%C','%H','%N','%O','%Metalloid','%Halogen','%Ametal']
work=df[features+[target]].copy()
work['LCD_PLD_ratio']=work['LCD (Å)']/work['PLD (Å)']
work['heteroatom_%']=work[['%N','%O','%Metalloid','%Halogen','%Ametal']].sum(axis=1)
display(work.head())


## Missing、scaling 与 selection
Missing 不等于 0。Scaling 对 KNN、linear/logistic models 常重要，对树模型通常不是必须。先删除 target-derived、ID、不可获得或重复 feature，再考虑复杂选择方法。

### 02C 完成标准
能够产生明确的 `X`、`y`，并记录 feature 的来源、单位与处理方式。